# Creating and loading Collections

A **collection** is a saved, executable recipe that combines datasets for one modeling task. It carries three things:

- **members** — which datasets will be part of this collection
- **a `Collection` chain** — how events are pooled, transformed, and assigned compatible data layouts
- **declared streams and targets** — what the collection reads and what the task decodes

The final `AssignDataLayouts` transform makes layout assignment part of running the collection: successful execution returns events that are already annotated and validated.

This notebook builds a `Collection` using the Python API, saves it to the database, reads it back, and inspects its event-native layouts.

> **Sections 4 onwards need the data drive.** `NEURALSHED_DATA_ROOT` must point at a mount holding `card_2024`, `willet_2023` and `fan_2023`. Everything up to and including saving runs against the catalog alone.

## The three calls

```
  save_collection(...)     cheap, no data file opened      →  a row in `collections`
  load_chain(...)          restores a runnable Collection  →  paths resolved locally
  collection.run()         loads and transforms events     →  annotated, validated layouts
```

---
## 0. Set up the database

A collection is a row in the metadata database, so we need a database instance running. Two commands set it up:

In [1]:
%%bash

# create the schema
alembic -c ../../src/neuralshed/database/alembic.ini upgrade head

# fill in datasets and event types
python -m neuralshed.database.populate --catalog-only

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will as

sume non-transactional DDL.


populating 15 dataset(s) into sqlite:////Users/luiztauffer/Github/neuralshed/src/neuralshed/

database/neuralshed_metadata.db
  - card_2024: catalog ok (catalog-only)
  - deo_2024: catalog ok (catal

og-only)
  - fan_2023: catalog ok (catalog-only)
  - fan_h1: catalog ok (catalog-only)
  - gusman_20

25: catalog ok (catalog-only)
  - jude_2026_speech: catalog ok (catalog-only)
  - jude_2026_typing: 

catalog ok (catalog-only)
  - karpowicz_2025: catalog ok (catalog-only)
  - kunz_2025: catalog ok (c

atalog-only)
  - singer_2025: catalog ok (catalog-only)
  - wairagkar_2025: catalog ok (catalog-only

)
  - willet_2021: catalog ok (catalog-only)
  - willet_2023: catalog ok (catalog-only)
  - willsey_

2025: catalog ok (catalog-only)
  - wilson_2025: catalog ok (catalog-only)
done.


The first command creates the database file if it does not exist yet, and builds the tables. The second reads each dataset's `metadata.py` and writes what it declares. Drop `--catalog-only` to also record one row per recorded session, which does need the drive mounted.

The cell below only reads: where the database lives, which datasets are available to put in a collection, and which collections already exist.

In [2]:
import json, time, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from sqlalchemy import inspect

import neuralset as ns
from neuralshed import collection, studies, transforms  # registers every step by name
from neuralshed.database import collections as dbc
from neuralshed.database import db, query

pd.set_option("display.width", 170, "display.max_columns", 40)

NAME = "speech-text-demo"

print("database:", db.database_url())
tables = set(inspect(db.get_engine()).get_table_names())

if "datasets" not in tables:
    print("\nno schema yet: run the two commands above, then re-run this cell.")
    summary, saved = pd.DataFrame(), pd.DataFrame()
else:
    summary, saved = query.catalog_summary(), dbc.list_collections()
    print(f"{len(summary)} datasets in the catalog | {len(saved)} collections saved\n")
    columns = ["name", "decodes", "n_datasets", "datasets"]
    print(saved[columns].to_string(index=False) if len(saved) else "(no collections yet)")

summary

database: sqlite:////Users/luiztauffer/Github/neuralshed/src/neuralshed/database/neuralshed_metadata.db
15 datasets in the catalog | 0 collections saved

(no collections yet)


,id,macro_task,source_data_platform,status,n_subjects,n_event_types,n_data_streams,n_sessions
0,card_2024,human_communication,dryad,ml_ready,1,3,2,45
1,fan_2023,human_communication,dryad,ml_ready,1,2,1,26
2,jude_2026_speech,human_communication,dryad,ml_ready,1,12,5,14
3,jude_2026_typing,human_communication,dryad,ml_ready,2,12,5,19
4,kunz_2025,human_communication,dryad,ml_ready,4,21,2,41
5,wairagkar_2025,human_communication,dryad,ml_ready,1,5,3,37
6,willet_2021,human_communication,dryad,ml_ready,1,7,1,11
7,willet_2023,human_communication,dryad,ml_ready,1,8,5,24
8,deo_2024,human_motor,dryad,ml_ready,1,5,3,13
9,fan_h1,human_motor,dandi,ml_ready,1,10,7,40


---
## 1. Declare the collection input and target events

`input_event_types` and `target_event_types` are neuroai **event class names**, the same vocabulary the catalog uses. They follow two different rules:

- **Inputs are wanted-if-present.** For example, `fan_2023` has no `BandPower` but can still belong in a spike-plus-band-power collection. It contributes one input stream instead of two.
- **Targets are required.** For example, a member with no `Sentence` contributes no trials at all, so it is either the wrong member or the wrong target.

`check_declared_streams` checks for both rules from the `event_types` catalog. 

In [3]:
INPUTS  = ("SpikeCounts", "BandPower")
TARGETS = ("Sentence",)
MEMBERS = ["card_2024", "willet_2023", "fan_2023"]

report = dbc.check_declared_streams(
    dataset_ids=MEMBERS,
    input_event_types=INPUTS,
    target_event_types=TARGETS,
)

pd.DataFrame({
    "Events available": {k: ", ".join(v) for k, v in report["available"].items()},
    "Declared inputs found": {k: ", ".join(v) for k, v in report["inputs"].items()},
})

,Events available,Declared inputs found
card_2024,"BandPower, Sentence, SpikeCounts","BandPower, SpikeCounts"
willet_2023,"BandPower, Sentence, SpikeCounts, Stimulus","BandPower, SpikeCounts"
fan_2023,"Sentence, SpikeCounts",SpikeCounts


`fan_2023` supplies `SpikeCounts` but not `BandPower`, and that is fine. What would not be fine is a member supplying none of the declared inputs, or no declared target, for example:

In [4]:
print("missing every declared input:", report["missing_inputs"])
print("missing the declared target:", report["missing_targets"])

bad = dbc.check_declared_streams(
    dataset_ids=["card_2024", "singer_2025", "fan_h1"],
    input_event_types=INPUTS,
    target_event_types=TARGETS,
)
print("\nsinger_2025 carries:\n", ", ".join(bad["available"]["singer_2025"]))
print("\nfan_h1 carries:     \n", ", ".join(bad["available"]["fan_h1"]))
print("\nmissing every declared input:", bad["missing_inputs"])
print("missing the declared target:", bad["missing_targets"])

missing every declared input: []
missing the declared target: []

singer_2025 carries:
 Action, BandPower, ClickAssistSeries, ClickStateSeries, CursorAssistSeries, CursorPositionSeries, DecodedCursorVelocitySeries, Intertrial, SpikeCounts, Stimulus, TargetPositionSeries, Word

fan_h1 carries:     
 Action, AngularVelocitySeries, GraspSeries, GraspVelocitySeries, Intertrial, LinearVelocitySeries, OrientationSeries, PositionSeries, Spikes, Stimulus

missing every declared input: ['fan_h1']
missing the declared target: ['fan_h1', 'singer_2025']


`singer_2025` decodes single `Word`s rather than sentences. `fan_h1` stores spike *times* (`Spikes`) rather than binned counts, so a collection reading `SpikeCounts` cannot read it at all. Both are caught before anything is written.

---
## 2. Compose the chain

The recipe is stored as a plain config dictionary and loads as a local `Collection(ns.Chain)`. A valid `Collection` starts with `CombineStudies`, the fan-in that pools members, and ends with exactly one `AssignDataLayouts`.

The stored form carries no `path`. A collection is portable and data directories are per machine, so paths are injected at load time.

Transforms can be applied to collection members or to the pooled frame, for example:

| step | writes | why |
|---|---|---|
| `StandardizeText` | `canonical_text` | normalized to exactly the characters the vocabulary can encode |
| `ValidateTarget` | `target_is_valid` | which rows are decode targets, as a mask rather than a filter |
| `AssignHashSplit` | `split` | train/val/test, assigned before any statistic is fitted |
| `AssignDataLayouts` | `data_layout_key`, `data_role` | explicit adapter identity and input/target roles |

Each member is keyed by its **member key**, and `CombineStudies` writes that key into a `collection_member` column.

In [5]:
def member(study_class, query=None):
    """One member: a Study step, optionally queried. A *list* of steps chains per-member transforms."""
    step = {"name": study_class}
    if query:
        step["query"] = query
    return step


QUERY = "session_index < 2"

CHAIN_CONFIG = {
    "name": "Collection",
    "steps": {
        "CombineStudies": {
            "name": "CombineStudies",
            "chains": {
                "card_2024":   member("Card2024Brain2Text", QUERY),
                "willet_2023": member("Willett2023Sentences", QUERY),
                "fan_2023":    member("Fan2023Handwriting", QUERY),
            },
        },
        "StandardizeText": {"name": "StandardizeText",
                            "source": "text", "dest": "canonical_text", "scheme": "v1"},
        "ValidateTarget":  {"name": "ValidateTarget",
                            "condition": "type == 'Sentence' and canonical_text != ''"},
        "AssignHashSplit": {"name": "AssignHashSplit", "by": ["timeline", "start"],
                            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1}, "seed": 0.0},
        "AssignDataLayouts": {
            "name": "AssignDataLayouts", "inputs": INPUTS, "targets": TARGETS
        },
    },
}

print(json.dumps(CHAIN_CONFIG, indent=1))

{
 "name": "Collection",
 "steps": {
  "CombineStudies": {
   "name": "CombineStudies",
   "chains": {
    "card_2024": {
     "name": "Card2024Brain2Text",
     "query": "session_index < 2"
    },
    "willet_2023": {
     "name": "Willett2023Sentences",
     "query": "session_index < 2"
    },
    "fan_2023": {
     "name": "Fan2023Handwriting",
     "query": "session_index < 2"
    }
   }
  },
  "StandardizeText": {
   "name": "StandardizeText",
   "source": "text",
   "dest": "canonical_text",
   "scheme": "v1"
  },
  "ValidateTarget": {
   "name": "ValidateTarget",
   "condition": "type == 'Sentence' and canonical_text != ''"
  },
  "AssignHashSplit": {
   "name": "AssignHashSplit",
   "by": [
    "timeline",
    "start"
   ],
   "ratios": {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
   },
   "seed": 0.0
  },
  "AssignDataLayouts": {
   "name": "AssignDataLayouts",
   "inputs": [
    "SpikeCounts",
    "BandPower"
   ],
   "targets": [
    "Sentence"
   ]
  }
 }
}


A config is just metadata until something loads it. `inject_paths` gives each member its data directory and `ns.Step(**cfg)` type-checks the result. Note that `data_dirs` is keyed by **study class**.

`ns.Step(**cfg)` is a discriminated loader keyed on `name`, so it returns the registered subclass named by the config, here `Collection`. The class checks the required first and final steps immediately.

In [6]:
PATHS = dbc.data_dirs(MEMBERS, must_exist=False)
for study_class, path in PATHS.items():
    print(f"{study_class:22s} {path}")

chain = ns.Step(**dbc.inject_paths(CHAIN_CONFIG, PATHS))
print("\nloads as", type(chain).__name__, "->", list(chain.steps))

Card2024Brain2Text     /Volumes/seagate_0/tether/datasets/card_2024/data
Fan2023Handwriting     /Volumes/seagate_0/tether/datasets/fan_2023/data
Willett2023Sentences   /Volumes/seagate_0/tether/datasets/willet_2023/data

loads as Collection -> ['CombineStudies', 'StandardizeText', 'ValidateTarget', 'AssignHashSplit', 'AssignDataLayouts']


---
## 3. Save a collection to the database

`save_collection` creates or replaces by name, so re-running this function upserts rather than duplicates. It runs `check_declared_streams` itself and refuses a member that cannot supply the declared target. The same `INPUTS` constant is used by the saved catalog declaration and by `AssignDataLayouts`, keeping the current persistence API aligned with the executable recipe.

In [7]:
collection_id = dbc.save_collection(
    collection_name=NAME,
    members=MEMBERS,
    description="Attempted speech and handwriting, spiking activity to sentence text.",
    chain=CHAIN_CONFIG,
    input_event_types=INPUTS,
    target_event_types=TARGETS,
)
dbc.list_collections()[["name", "decodes", "n_datasets", "datasets"]]

,name,decodes,n_datasets,datasets
0,speech-text-demo,Sentence,3,"card_2024, fan_2023, willet_2023"


If one of the proposed members for a collection cannot supply the target event type, it never reaches the database at all.

In [8]:
try:
    dbc.save_collection(
        collection_name="never-saved",
        members=["card_2024", "singer_2025"],
        chain=CHAIN_CONFIG,
        input_event_types=INPUTS,
        target_event_types=TARGETS,
    )
except ValueError as e:
    print("refused:", e)

print("\nin the database:", dbc.get_collection(collection_name="never-saved"))

refused: members cannot supply the declared targets ['Sentence']: ['singer_2025']; targets are required

in the database: None


---
## 4. Read a collection from the database

`get_collection` returns the stored record as a dict. `load_chain` restores a **runnable** `Collection` with paths resolved against this machine. Running it applies every transform and performs layout validation at the end.

In [9]:
col = dbc.get_collection(collection_name=NAME)

print("datasets   :", ", ".join(col["datasets"]))
print("reads      :", col["input_event_types"])
print("decodes    :", col["target_event_types"])
print("description:", col["description"])

datasets   : card_2024, fan_2023, willet_2023
reads      : ['SpikeCounts', 'BandPower']
decodes    : ['Sentence']
description: Attempted speech and handwriting, spiking activity to sentence text.


Now restore the executable recipe and run it. The final layout transform may inspect channel headers or one representative array when resolving widths, but the event table remains the collection boundary.

In [10]:
t0 = time.time()
chain = dbc.load_chain(collection_name=NAME, must_exist=True)
events = chain.run().reset_index(drop=True)
print(f"{len(events)} events in {time.time() - t0:.1f}s")

events.groupby(["collection_member", "type"]).size().unstack(fill_value=0)

4451 events in 66.8s


type,BandPower,Sentence,SpikeCounts,Stimulus
collection_member,,,,
card_2024,706,671,706,0
fan_2023,0,130,130,0
willet_2023,34,680,34,1360


Two things to read out of that table. `fan_2023` has no `BandPower` row, the missing stream declared as wanted-if-present. And the `SpikeCounts` counts differ wildly from the `Sentence` counts because the members store timelines differently: `card_2024` and `fan_2023` write one timeline per trial, `willet_2023` one per recording block with 20-50 trials inside it.

All transforms have run, so the frame now carries target/split metadata and layout annotations. Dense arrays remain lazy until their channel width must be resolved or an extractor reads them.

In [11]:
from neuralshed.transforms import valid_targets

targets = valid_targets(events)
print(f"{len(targets)} valid targets\n")
print(targets.groupby(["collection_member", "split"]).size().unstack(fill_value=0).to_string())

targets[["collection_member", "session", "text", "canonical_text", "split"]].head(4)

1481 valid targets

split              test  train  val
collection_member                  
card_2024            82    537   52
fan_2023             16    101   13
willet_2023          73    529   78


,collection_member,session,text,canonical_text,split
2,card_2024,t15.2023.08.11,Bring it closer.,bring it closer,train
5,card_2024,t15.2023.08.11,My family is closer.,my family is closer,train
8,card_2024,t15.2023.08.11,What do they like?,what do they like,test
11,card_2024,t15.2023.08.11,How is that good?,how is that good,train


---
## 5. Inspect the data layouts

`AssignDataLayouts` marks declared events as `input`, `target`, or `both`. Every declared row also receives a `data_layout_key` selecting a compatible input or target adapter. A `Sentence` key routes text encoding and the model head; it does not imply channels.

Every dense event requires Study-authored `n_features`, the size of axis 0 before extraction. It validates an explicitly requested grouping but never changes the key unless the user includes `n_features` in `group_by`. Keys use named `column=value` components. If a grouping field is inapplicable to an entire event type it is omitted once with a warning; partial population raises. `frequency` and `ch_names` likewise remain source-event facts.

In [12]:
layout_columns = [
    "data_layout_key", "data_role", "type", "n_features",
    "participant_id", "collection_member", "session", "frequency",
]
layouts = (
    events.loc[events["data_layout_key"] != "", layout_columns]
    .drop_duplicates()
    .sort_values(["collection_member", "session", "type"])
    .reset_index(drop=True)
)
layouts

,data_layout_key,data_role,type,n_features,participant_id,collection_member,session,frequency
0,participant_id=braingate2%2FT15/collection_mem...,input,BandPower,256,braingate2/T15,card_2024,t15.2023.08.11,50.0
1,participant_id=braingate2%2FT15/collection_mem...,target,Sentence,<NA>,braingate2/T15,card_2024,t15.2023.08.11,NaN
2,participant_id=braingate2%2FT15/collection_mem...,input,SpikeCounts,256,braingate2/T15,card_2024,t15.2023.08.11,50.0
3,participant_id=braingate2%2FT15/collection_mem...,input,BandPower,256,braingate2/T15,card_2024,t15.2023.08.13,50.0
4,participant_id=braingate2%2FT15/collection_mem...,target,Sentence,<NA>,braingate2/T15,card_2024,t15.2023.08.13,NaN
5,participant_id=braingate2%2FT15/collection_mem...,input,SpikeCounts,256,braingate2/T15,card_2024,t15.2023.08.13,50.0
6,participant_id=braingate2%2FT5/collection_memb...,target,Sentence,<NA>,braingate2/T5,fan_2023,t5.2022.05.18,NaN
7,participant_id=braingate2%2FT5/collection_memb...,input,SpikeCounts,192,braingate2/T5,fan_2023,t5.2022.05.18,50.0
8,participant_id=braingate2%2FT5/collection_memb...,target,Sentence,<NA>,braingate2/T5,fan_2023,t5.2022.05.23,NaN
9,participant_id=braingate2%2FT5/collection_memb...,input,SpikeCounts,192,braingate2/T5,fan_2023,t5.2022.05.23,50.0


Collapsing the table by member and event type makes the different input widths easy to see.

In [13]:
layouts.groupby(["collection_member", "type"])["n_features"].unique().to_frame()

n_features
collection_member type                  
card_2024         BandPower        [256]
                  Sentence        [<NA>]
                  SpikeCounts      [256]
fan_2023          Sentence        [<NA>]
                  SpikeCounts      [192]
willet_2023       BandPower        [256]
                  Sentence        [<NA>]
                  SpikeCounts      [256]

A few details are visible immediately:

- A recording carrying spike counts, band power, and a sentence target has three model-facing layout keys; missing `BandPower` is represented by no row.
- The default key uses named `(participant_id, collection_member, session, event type)` components, so every key names one model-facing input or target layout.
- Speech streams have 256 channels while `fan_2023` has 192. Different widths are safe because they receive different adapters.

The `Sentence` rows carry `data_role="target"` and their own layout keys but no `n_features`; their keys can route the target extractor, vocabulary, output head, and loss.

### Layout policy is part of the recipe

The final transform stores the layout policy itself. Changing its `inputs`, `targets`, or `group_by` changes the serialized collection rather than making a separate layout artifact stale.

In [14]:
chain.layout_step.model_dump()

{'infra': None,
 'inputs': ('SpikeCounts', 'BandPower'),
 'targets': ('Sentence',),
 'group_by': ('participant_id', 'collection_member', 'session'),
 'name': 'AssignDataLayouts'}

Here the conservative default gives each member/session/event type its own adapter. A future collection can deliberately share adapters by choosing fewer `group_by` columns, while execution still rejects incompatibilities exposed by the source channel headers.

---
## 6. Validation happens during `run()`

Reaching this point means layout validation already passed. The summary below checks the two most important postconditions: every resolved key has exactly one type/role/width, and every source timeline contains at least one declared input.

In [15]:
layout_checks = events.loc[events["data_layout_key"] != ""].groupby("data_layout_key").agg(
    event_types=("type", "nunique"),
    roles=("data_role", "nunique"),
)
dense_widths = layouts.dropna(subset=["n_features"]).groupby("data_layout_key")["n_features"].nunique()
print("all layout signatures unique:", layout_checks.eq(1).all().all())
print("all dense widths consistent:", dense_widths.eq(1).all())

input_rows = events["data_role"].isin(["input", "both"])
input_recordings = events.loc[input_rows, ["collection_member", "timeline"]].drop_duplicates()
all_recordings = events[["collection_member", "timeline"]].drop_duplicates()
print("every recording has input data:", len(input_recordings) == len(all_recordings))

all layout signatures unique: True
all dense widths consistent: True
every recording has input data: True


Validation is automatic rather than a second method call. Required dense `n_features` values and available channel headers must agree within each explicitly requested layout. A mismatch raises instead of silently creating another key. Native rate is deliberately not a layout constraint because extractors may resample before model input.

In [16]:
print("layout keys :", layouts["data_layout_key"].nunique())
print("participants:", events["participant_id"].nunique())
print("recordings  :", len(all_recordings))

layout keys : 16
participants: 3
recordings  : 870


---
## 7. From here

The next boundary segments these annotated events into naturally sized samples:

```python
collection = dbc.load_chain(collection_name=NAME, must_exist=True)
events = collection.run()
```

[03_collection_data_pipeline.ipynb](03_collection_data_pipeline.ipynb) follows these events through ragged segmentation, split-aware batch sampling, PyTorch collation, and the packaged `BICData` build boundary. The current BIT bundle shown in tutorial 04 still consumes the older manifest compatibility path; migrating it to these event-native annotations is the next implementation step.

### The real collection

This notebook queried every member down to two sessions. Dropping `QUERY` gives the full thing, at a different scale but through the same calls:

```python
CHAIN_CONFIG["steps"]["CombineStudies"]["chains"] = {
    "card_2024":   member("Card2024Brain2Text"),      # no query: every session
    "willet_2023": member("Willett2023Sentences"),
    "fan_2023":    member("Fan2023Handwriting"),
}
dbc.save_collection(collection_name="speech-text", members=MEMBERS, chain=CHAIN_CONFIG,
                    input_event_types=INPUTS, target_event_types=TARGETS)
```

Loading and running that saved recipe reproduces and validates the full collection's layout annotations from its events.

### Housekeeping

```python
dbc.delete_collection(collection_name=NAME)   # removes the saved collection and its members
```